# Confidence feature selector lokalnie

Ten notebook dokumentuje i uruchamia kolejną lokalną iterację po `CONFIDENCE_AWARE_SELECTION_LOCAL.ipynb`.

Cel: sprawdzić, czy z mierzalnych cech zaufania/confidence da się wybrać wariant rekonstrukcji per obraz lepiej niż stały baseline `nn_eegnet_top1_gate`.

To nadal jest etap candidate-constrained reconstruction: model wybiera albo miesza obrazy z puli kandydatów. Nie jest to jeszcze wolna generacja obrazu z EEG od zera.

## Kontekst z poprzednich iteracji

Najważniejsze ustalenia przed tym eksperymentem:

- `candidate_zscore` ograniczył hubness, ale nie przebił VAE (`SSIM ≈ 0.264`).
- `eegnet_top1_category_gate` przebił VAE jako candidate-constrained reconstruction (`SSIM ≈ 0.308`).
- top-k prototype/blending osiągnął okolice VAE (`SSIM ≈ 0.286`), ale rozmywał szczegóły.
- prosty selektor po przewidzianej kategorii pogorszył wynik (`SSIM ≈ 0.292`).
- oracle per-obraz wśród tych samych kandydatów miał duży zapas (`SSIM ≈ 0.386`).

Hipoteza tej iteracji: problemem nie jest brak dobrych kandydatów, tylko brak dobrego predyktora, który mówi, któremu kandydatowi/metodzie zaufać.

## Co robi skrypt

`scripts/run_confidence_feature_selector.py` buduje tabelę kandydatów i cech dla walidacji oraz testu.

Kandydaci:

- `nn_candidate_zscore`,
- `nn_eegnet_top1_gate`,
- `nn_eegnet_zlogprob_1p5`,
- `nn_eegnet_zlogprob_2`,
- prototyp `eegnet_top1_category_gate_k3_t0.5_anchor0.5`.

Cechy confidence obejmują między innymi:

- marginesy score'ów retrieval,
- entropię score'ów top-k,
- hubness top-1 kandydata,
- prawdopodobieństwa i entropię EEGNet,
- zgodność kategorii EEGNet z kategoriami kandydatów,
- typ metody jako one-hot.

Na walidacji uczymy Ridge regresję przewidującą realny `SSIM` kandydata. Alpha jest wybierana przez leave-one-image-out CV na walidacji. Test służy dopiero do końcowej oceny. Skrypt liczy też wariant NN-only oraz oracle upper bound.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'scripts').is_dir() else CWD.parent
OUTPUT_DIR = PROJECT_ROOT / 'wyniki colab' / 'confidence_feature_selector_mole_local'
SCRIPT = PROJECT_ROOT / 'scripts' / 'run_confidence_feature_selector.py'

required_paths = {
    'script': SCRIPT,
    'category-aware results': PROJECT_ROOT / 'wyniki colab' / 'category_aware_reranking_mole_local',
    'prototype results': PROJECT_ROOT / 'wyniki colab' / 'category_prototype_reconstruction_mole_local',
    'manifest': PROJECT_ROOT / 'reconstruction_manifests' / 'participant_image_mole_no_abc',
    'embeddings': PROJECT_ROOT / 'image_embeddings_unclip_participant_image_mole_no_abc_local_20260627',
    'retrieval checkpoint': PROJECT_ROOT / 'wyniki colab' / 'unclip_mole_retrieval' / 'eeg_image_retrieval.pt',
    'eegnet checkpoint': PROJECT_ROOT / 'wyniki colab' / 'eegnet_mole_colab' / 'eegnet.pt',
}

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR   =', OUTPUT_DIR)

missing = []
for name, path in required_paths.items():
    ok = path.exists()
    print(f'{name:24s}', 'OK' if ok else 'BRAK', path)
    if not ok:
        missing.append((name, path))

if missing:
    raise FileNotFoundError('Brakuje wymaganych plików/katalogów: ' + ', '.join(name for name, _ in missing))

In [ ]:
# Uruchom pełny lokalny eksperyment.
# --force czyści tylko OUTPUT_DIR tej iteracji, żeby nie mieszać starych wyników.
import subprocess
import sys

cmd = [
    sys.executable, str(SCRIPT),
    '--project-root', str(PROJECT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Główne porównanie selektorów na teście.
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / 'confidence_feature_selector_summary.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(OUTPUT_DIR / 'selector_comparison.csv')

display(comparison.sort_values('ssim', ascending=False))

print('Najlepsza alpha all-method CV:', summary['best_alpha_by_validation_leave_one_image_cv'])
print('Najlepsza alpha NN-only CV:', summary['best_alpha_nn_only_by_validation_leave_one_image_cv'])
print('Globalny baseline wybrany walidacyjnie:', summary['global_validation_best_method'])

In [ ]:
# Walidacyjna leave-one-image-out CV.
cv_all = pd.read_csv(OUTPUT_DIR / 'validation_leave_one_image_cv.csv')
cv_nn = pd.read_csv(OUTPUT_DIR / 'validation_leave_one_image_cv_nn_only.csv')

print('All-method selector:')
display(cv_all)

print('NN-only selector:')
display(cv_nn)

In [ ]:
# Podgląd tabel cech confidence.
for filename in ['validation_confidence_features.csv', 'test_confidence_features.csv']:
    path = OUTPUT_DIR / filename
    df = pd.read_csv(path)
    print(filename, df.shape)
    display(df.head())
    print()

In [ ]:
# Podgląd gridów: target vs predykcja dla selektorów i oracle.
from IPython.display import Image as IPImage, Markdown, display

grid_dir = OUTPUT_DIR / 'grids'
for path in sorted(grid_dir.glob('*.jpg')):
    display(Markdown(f'### {path.name}'))
    display(IPImage(filename=str(path)))

## Wpis historyczny — wynik lokalny z 2026-06-27

Wynik jest diagnostycznie ważny, ale nie poprawia aktualnego najlepszego baseline'u.

Najważniejsze liczby na teście `mole`:

- globalny baseline `nn_eegnet_top1_gate`: `SSIM = 0.308`, `L1 = 0.261`,
- Ridge all-method confidence selector: `SSIM = 0.304`, `L1 = 0.240`,
- Ridge NN-only confidence selector: `SSIM = 0.300`, `L1 = 0.266`,
- oracle NN-only per-obraz: `SSIM = 0.361`,
- oracle all-method per-obraz: `SSIM = 0.386`.

Na walidacji leave-one-image-out wynik wyglądał obiecująco (`SSIM ≈ 0.343`), ale na teście selektor spadł poniżej prostego globalnego baseline'u. All-method selector zbyt często wybierał prototyp, a NN-only selector nie nauczył się stabilnie kiedy odejść od `nn_eegnet_top1_gate`.

Wniosek: obecne cechy confidence są użyteczne diagnostycznie, ale nie są jeszcze wystarczająco stabilnym mechanizmem wyboru końcowej rekonstrukcji. Najlepszy uczciwy wynik pozostaje `nn_eegnet_top1_gate` (`SSIM ≈ 0.308`). Następny sensowny krok to więcej danych kalibracyjnych/splitów albo trening predyktora confidence na większej liczbie uczestników, zamiast dokładania kolejnych ręcznych reguł na jednym małym zbiorze.